# 1. Import Library dan Setup
Bagian ini digunakan untuk mengimpor semua library yang dibutuhkan untuk proses pemodelan, seperti `scikit-learn` untuk preprocessing/split, `xgboost`, dan `tensorflow.keras` untuk deep learning.

In [1]:
import numpy as np
import scipy.sparse as sp
import joblib
import os
import tensorflow as tf
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Bidirectional, Conv1D, GlobalMaxPooling1D, Dropout, Embedding
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')


# 2. Load Extracted Features dan Labels
Membaca fitur-fitur yang telah diekstrak pada tahapan sebelumnya (TF-IDF, Word2Vec, GloVe, dan FastText) beserta target labels dari folder `extracted_features/`.

In [2]:
# Load data
feature_dir = 'extracted_features'
# Memuat label
y = np.load(os.path.join(feature_dir, 'y_labels.npy'))
# Memuat semua fitur
X_tfidf = sp.load_npz(os.path.join(feature_dir, 'X_tfidf.npz')) # Sparse matrix
X_seq = np.load(os.path.join(feature_dir, 'X_seq.npy'))
# Load embedding matrices
emb_w2v = np.load(os.path.join(feature_dir, 'embedding_matrix_w2v.npy'))
emb_ft = np.load(os.path.join(feature_dir, 'embedding_matrix_ft.npy'))
emb_glove = np.load(os.path.join(feature_dir, 'embedding_matrix_glove.npy'))
print(f"Bentuk TF-IDF    : {X_tfidf.shape}")
print(f"Bentuk Sequence  : {X_seq.shape}")
print(f"Bentuk Label     : {y.shape}")


Bentuk TF-IDF    : (20000, 5016)
Bentuk Sequence  : (20000, 200)
Bentuk Label     : (20000,)


# 3. Split Data (Train & Test)
Membagi masing-masing set fitur menjadi data latih (80%) dan data uji (20%) menggunakan parameter `random_state` yang sama agar pembagian observasinya konsisten pada setiap eksperimen.

In [3]:
# Kita akan menggunakan Stratified K-Fold CV untuk evaluasi ML Klasik, 
# tapi kita juga siapkan train_test_split untuk training Deep Learning
X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, stratify=y, random_state=42)
X_train_seq, X_test_seq, _, _ = train_test_split(X_seq, y, test_size=0.2, stratify=y, random_state=42)
print(f"Train TF-IDF: {X_train_tfidf.shape}, Test TF-IDF: {X_test_tfidf.shape}")
print(f"Train Sequence: {X_train_seq.shape}, Test Sequence: {X_test_seq.shape}")


Train TF-IDF: (16000, 5016), Test TF-IDF: (4000, 5016)
Train Sequence: (16000, 200), Test Sequence: (4000, 200)


# 4. Fungsi Evaluasi Model
Mendefinisikan fungsi pembantu untuk menyimpan log performa, agar nantinya dapat digunakan pada evaluasi akhir keseluruhan model.

In [4]:
results = []

def evaluate_model(model_name, feature_name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    results.append({
        'Model': model_name,
        'Feature': feature_name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })
    
    print(f"\n[{model_name} | {feature_name}]")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-Score  : {f1:.4f}")

# 5. Pemodelan dengan XGBoost
Algoritma berbasis tree-ensemble ini cukup handal dalam klasifikasi. XGBoost akan dilatih dengan keempat varian representasi teks.

In [5]:
# Evaluasi ML Klasik (Random Forest & XGBoost) dengan Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
ml_models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}
for model_name, model in ml_models.items():
    print(f"Evaluating {model_name} with 5-Fold CV...")
    acc_scores, f1_scores = [], []
    for train_idx, test_idx in skf.split(X_tfidf, y):
        X_train_cv, X_test_cv = X_tfidf[train_idx], X_tfidf[test_idx]
        y_train_cv, y_test_cv = y[train_idx], y[test_idx]
        
        model.fit(X_train_cv, y_train_cv)
        y_pred_cv = model.predict(X_test_cv)
        
        acc_scores.append(accuracy_score(y_test_cv, y_pred_cv))
        f1_scores.append(f1_score(y_test_cv, y_pred_cv, average='weighted', zero_division=0))
        
    avg_acc = np.mean(acc_scores)
    avg_f1 = np.mean(f1_scores)
    
    results.append({
        'Model': model_name,
        'Feature': 'TF-IDF + Metadata',
        'Accuracy': avg_acc,
        'Precision': 0.0, # Simplified for CV output
        'Recall': 0.0,
        'F1-Score': avg_f1
    })
    
    print(f"[{model_name}] Avg Accuracy: {avg_acc:.4f}, Avg F1-Score: {avg_f1:.4f}")


Evaluating Random Forest with 5-Fold CV...


[Random Forest] Avg Accuracy: 0.5046, Avg F1-Score: 0.5044
Evaluating XGBoost with 5-Fold CV...


[XGBoost] Avg Accuracy: 0.4999, Avg F1-Score: 0.4998


# 6. Pemodelan dengan CNN (Convolutional Neural Network)
CNN dapat menangkap relasi spasial dalam sequence data. Untuk Deep Learning (CNN dan LSTM) yang mensyaratkan input sequence 3D `(batch_size, sequence_length, embedding_dim)`, kita akan mereshape matriks fitur teks (berdimensi `N x Dimensi_Fitur`) menjadi sekumpulan sequence.

In [6]:
def create_cnn_model(embedding_matrix, input_length):
    vocab_size, embedding_dim = embedding_matrix.shape
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embedding_dim, 
                  weights=[embedding_matrix], input_length=input_length, trainable=False),
        Conv1D(filters=64, kernel_size=3, padding='same', activation='relu'),
        GlobalMaxPooling1D(),
        Dense(32, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
embeddings = {
    'Word2Vec': emb_w2v,
    'FastText': emb_ft,
    'GloVe': emb_glove
}
for emb_name, emb_matrix in embeddings.items():
    print(f"Training CNN with {emb_name} embeddings...")
    model = create_cnn_model(emb_matrix, X_train_seq.shape[1])
    
    model.fit(X_train_seq, y_train, epochs=10, batch_size=32, 
              validation_split=0.2, callbacks=[early_stopping], verbose=0)
    
    y_pred_probs = model.predict(X_test_seq, verbose=0)
    y_pred = (y_pred_probs > 0.5).astype(int).flatten()
    
    evaluate_model('CNN', emb_name, y_test, y_pred)


Training CNN with Word2Vec embeddings...



[CNN | Word2Vec]
Accuracy  : 0.5045
Precision : 0.5133
Recall    : 0.5045
F1-Score  : 0.3698
Training CNN with FastText embeddings...



[CNN | FastText]
Accuracy  : 0.5055
Precision : 0.5115
Recall    : 0.5055
F1-Score  : 0.4546
Training CNN with GloVe embeddings...



[CNN | GloVe]
Accuracy  : 0.5028
Precision : 0.2528
Recall    : 0.5028
F1-Score  : 0.3364


# 7. Pemodelan dengan LSTM (Long Short-Term Memory)
LSTM sangat tangguh dalam menangkap dependensi jangka panjang pada sequence data. Kita akan memperlakukan fitur seperti sequence time-step agar cocok dengan layer LSTM.

In [7]:
def create_bilstm_model(embedding_matrix, input_length):
    vocab_size, embedding_dim = embedding_matrix.shape
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embedding_dim, 
                  weights=[embedding_matrix], input_length=input_length, trainable=False),
        Bidirectional(LSTM(64, return_sequences=False)),
        Dropout(0.5),
        Dense(32, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model
for emb_name, emb_matrix in embeddings.items():
    print(f"Training Bi-LSTM with {emb_name} embeddings...")
    model = create_bilstm_model(emb_matrix, X_train_seq.shape[1])
    
    model.fit(X_train_seq, y_train, epochs=10, batch_size=32, 
              validation_split=0.2, callbacks=[early_stopping], verbose=0)
    
    y_pred_probs = model.predict(X_test_seq, verbose=0)
    y_pred = (y_pred_probs > 0.5).astype(int).flatten()
    
    evaluate_model('Bi-LSTM', emb_name, y_test, y_pred)


Training Bi-LSTM with Word2Vec embeddings...



[Bi-LSTM | Word2Vec]
Accuracy  : 0.4915
Precision : 0.4898
Recall    : 0.4915
F1-Score  : 0.4472
Training Bi-LSTM with FastText embeddings...



[Bi-LSTM | FastText]
Accuracy  : 0.4945
Precision : 0.4939
Recall    : 0.4945
F1-Score  : 0.4415
Training Bi-LSTM with GloVe embeddings...



[Bi-LSTM | GloVe]
Accuracy  : 0.5025
Precision : 0.5012
Recall    : 0.5025
F1-Score  : 0.4427


In [8]:
# simpan semua model LSTM, XGBoost, dan CNN dengan format .pkl

# 8. Evaluasi dan Perbandingan Keseluruhan
Menyusun semua hasil eksperimen ke dalam Pandas DataFrame untuk membandingkan secara eksplisit performa masing-masing model yang dikombinasikan dengan jenis ekstraksi fitur yang berbeda.

In [9]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=['F1-Score', 'Accuracy'], ascending=False).reset_index(drop=True)

print("=== PERBANDINGAN MODEL DAN FEATURE EXTRACTION ===")
display(results_df)

=== PERBANDINGAN MODEL DAN FEATURE EXTRACTION ===


,Model,Feature,Accuracy,Precision,Recall,F1-Score
0,Random Forest,TF-IDF + Metadata,0.50455,0.000000,0.00000,0.504378
1,XGBoost,TF-IDF + Metadata,0.49990,0.000000,0.00000,0.499831
2,CNN,FastText,0.50550,0.511525,0.50550,0.454604
3,Bi-LSTM,Word2Vec,0.49150,0.489760,0.49150,0.447223
4,Bi-LSTM,GloVe,0.50250,0.501236,0.50250,0.442715
5,Bi-LSTM,FastText,0.49450,0.493857,0.49450,0.441528
6,CNN,Word2Vec,0.50450,0.513270,0.50450,0.369792
7,CNN,GloVe,0.50275,0.252758,0.50275,0.336393


# 9. Menyimpan Model Terbaik
Dari hasil perbandingan di atas, model beserta jenis representasi vektornya yang memberikan F1-Score atau Accuracy terbaik akan disimpan di dalam direktori `Model/` untuk dapat digunakan pada saat deployment (Gradio / Streamlit).

In [10]:
# Evaluasi ML Klasik (Random Forest & XGBoost) dengan Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
ml_models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}
for model_name, model in ml_models.items():
    print(f"Evaluating {model_name} with 5-Fold CV...")
    acc_scores, f1_scores = [], []
    for train_idx, test_idx in skf.split(X_tfidf, y):
        X_train_cv, X_test_cv = X_tfidf[train_idx], X_tfidf[test_idx]
        y_train_cv, y_test_cv = y[train_idx], y[test_idx]
        
        model.fit(X_train_cv, y_train_cv)
        y_pred_cv = model.predict(X_test_cv)
        
        acc_scores.append(accuracy_score(y_test_cv, y_pred_cv))
        f1_scores.append(f1_score(y_test_cv, y_pred_cv, average='weighted', zero_division=0))
        
    avg_acc = np.mean(acc_scores)
    avg_f1 = np.mean(f1_scores)
    
    results.append({
        'Model': model_name,
        'Feature': 'TF-IDF + Metadata',
        'Accuracy': avg_acc,
        'Precision': 0.0, # Simplified for CV output
        'Recall': 0.0,
        'F1-Score': avg_f1
    })
    
    print(f"[{model_name}] Avg Accuracy: {avg_acc:.4f}, Avg F1-Score: {avg_f1:.4f}")


Evaluating Random Forest with 5-Fold CV...


[Random Forest] Avg Accuracy: 0.5046, Avg F1-Score: 0.5044
Evaluating XGBoost with 5-Fold CV...


[XGBoost] Avg Accuracy: 0.4999, Avg F1-Score: 0.4998
